# PolyGuard 최소 inference 데모

`text -> risk -> category -> confidence -> reason` 을 PolyGuard-Qwen-Smol 모델 하나로만 구현한다.
기존 `models.py`의 `load()`/`moderate()`를 그대로 재사용한다 (파싱/confidence 로직 중복 방지).

In [ ]:
# conda activate guard 환경에서 실행
from models import load, moderate

MODEL_ID = "ToxicityPrompts/PolyGuard-Qwen-Smol"
tok, model = load(MODEL_ID)

In [ ]:
# POLYGUARD_SYSTEM_PROMPT(models.py)의 S1-S14 정의를 축약한 표시용 이름
CATEGORY_NAMES = {
    "S1": "Violent Crimes",
    "S2": "Non-Violent Crimes",
    "S3": "Sex Crimes",
    "S4": "Child Exploitation",
    "S5": "Defamation",
    "S6": "Specialized Advice",
    "S7": "Privacy",
    "S8": "Intellectual Property",
    "S9": "Indiscriminate Weapons",
    "S10": "Hate",
    "S11": "Self-Harm",
    "S12": "Sexual Content",
    "S13": "Elections",
    "S14": "Code Interpreter Abuse",
}


def infer(text):
    """text -> {text, risk_category_confidence, reason} 딕셔너리 반환."""
    result = moderate("polyguard", tok, model, text)
    pct = round(result["confidence"] * 100) if result["confidence"] is not None else 0

    if result["risk"] == "unsafe" and result["category"]:
        label = CATEGORY_NAMES.get(result["category"][0], result["category"][0])
    else:
        label = "Safe"

    return {
        "text": text,
        "risk_category_confidence": f"{label} (Confidence: {pct}%)",
        "reason": result["reason"],
    }

In [ ]:
infer("이 나쁜 녀석아!")

In [ ]:
infer("오늘 날씨 어때?")